# Frontend-stack specialist — Qwen3-8B QLoRA

Self-contained Colab workflow: the frozen dataset, tool schemas, and pinned chat template are embedded in this notebook. It does **not** clone GitHub. Google Drive is opt-in, so a failed Drive mount cannot block training.

Choose **Runtime → Change runtime type → GPU**, then **Run all**. A T4 (16 GB) is the minimum; L4/A100 is better. The default uses temporary Colab storage and automatically starts the final GGUF download. Set `USE_GOOGLE_DRIVE = True` in cell 3 only when Drive mounting works.


In [ ]:
# Install the same working stack used by the local hardware probes.
%pip install -q --upgrade --no-cache-dir "unsloth==2026.5.8" "unsloth_zoo==2026.5.4" "transformers==4.57.6" "trl==0.23.1" "bitsandbytes==0.49.2" "datasets==4.3.0" "peft==0.18.1" "accelerate==1.13.0" "huggingface_hub==0.36.2" "tokenizers==0.22.2" "safetensors==0.7.0"

In [ ]:
import json, os, platform, subprocess, time
from pathlib import Path
# Apply Unsloth patches before importing torch/transformers/TRL. Conservative chunking lowers peak VRAM.
os.environ.setdefault('UNSLOTH_CE_LOSS_TARGET_GB', '0.05')
os.environ.setdefault('UNSLOTH_DISABLE_DOUBLE_BUFFER', '1')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import unsloth
import torch
from importlib.metadata import version
EXPECTED_PACKAGES = {'unsloth': '2026.5.8', 'unsloth_zoo': '2026.5.4', 'transformers': '4.57.6', 'trl': '0.23.1', 'bitsandbytes': '0.49.2', 'datasets': '4.3.0', 'peft': '0.18.1', 'accelerate': '1.13.0', 'huggingface_hub': '0.36.2', 'tokenizers': '0.22.2', 'safetensors': '0.7.0'}
PACKAGE_VERSIONS = {name: version(name) for name in EXPECTED_PACKAGES}
if PACKAGE_VERSIONS != EXPECTED_PACKAGES: raise RuntimeError(f'Package version mismatch: expected {EXPECTED_PACKAGES}, got {PACKAGE_VERSIONS}')
print('Pinned packages:', PACKAGE_VERSIONS)

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. Select Runtime → Change runtime type → GPU, then run all.')
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 1024**3
print(f'GPU: {gpu.name} | VRAM: {vram_gib:.2f} GiB | CUDA: {torch.version.cuda}')
if vram_gib < 14.5:
    raise RuntimeError(f'{vram_gib:.2f} GiB is insufficient for the guarded 4K run; reconnect to a T4/L4/A100 runtime.')

# Keep this False when Drive authentication is unreliable. Training then runs without any mount prompt.
USE_GOOGLE_DRIVE = False
NOTEBOOK_REVISION = '2026-08-06-self-contained-v8-gguf-fix'
DRIVE_MOUNT = Path('/content/drive')
DRIVE_AVAILABLE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    try:
        drive.mount(str(DRIVE_MOUNT), force_remount=True, timeout_ms=300_000)
        DRIVE_AVAILABLE = (DRIVE_MOUNT / 'MyDrive').is_dir()
        if not DRIVE_AVAILABLE: raise RuntimeError('Drive reported success but MyDrive is unavailable')
    except Exception as mount_error:
        print(f'WARNING: Google Drive mount failed: {mount_error}')
        print('Continuing in temporary runtime storage; no retry or crash.')
DRIVE_ROOT = ((DRIVE_MOUNT / 'MyDrive') if DRIVE_AVAILABLE else Path('/content')) / 'Master-Models-Colab/frontend-stack-qwen3-8b'
TRAINER_OUT = DRIVE_ROOT / 'trainer'
LORA_OUT = DRIVE_ROOT / 'lora'
GGUF_OUT = DRIVE_ROOT / 'gguf'
for path in (TRAINER_OUT, LORA_OUT, GGUF_OUT): path.mkdir(parents=True, exist_ok=True)
print(('Persistent Drive output:' if DRIVE_AVAILABLE else 'TEMPORARY runtime output:'), DRIVE_ROOT)
print('Notebook revision:', NOTEBOOK_REVISION)


In [ ]:
# Restore the frozen project payload embedded in this notebook. No git clone or network archive is used.
import base64, hashlib, io, shutil, zipfile
REPO_COMMIT = '0de1e5e87782ec6ec4ece287aa93ba80e20cadfd'
WORK = Path('/content/Master-Models')
PAYLOAD_SHA256 = 'a6e21c5330d2ccf4e49bcade42edc37d3a93957621943c781a0df6ea29e4d283'
PAYLOAD_HASHES = {"datasets/frontend-stack/final/holdout.jsonl": "8d59b5c22ee6df65d8b1843d0bdf08dc352aa6640efee07d8fc6d17158485372", "datasets/frontend-stack/final/train.jsonl": "f4c36ca2b83d98137841777eb2d64024d929c82fdacfad75292607be5777d19d", "training/pi_tools.json": "72190fac6d837652b56944b4f5891aae6c882ce045a5945e3a1b0b1813241b43", "training/templates/qwen3-8b.jinja": "0958f6ebb716badab20903885efa1a26ee5a8d2ce9162bfcfe982ff43cff4077"}
PAYLOAD_B64 = '''...'''

payload = base64.b64decode(''.join(PAYLOAD_B64.split()), validate=True)
if hashlib.sha256(payload).hexdigest() != PAYLOAD_SHA256:
    raise RuntimeError('Embedded payload checksum mismatch; upload a fresh notebook copy.')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    expected = set(PAYLOAD_HASHES)
    actual = {info.filename for info in archive.infolist() if not info.is_dir()}
    if actual != expected: raise RuntimeError(f'Embedded payload file list mismatch: {actual ^ expected}')
    for info in archive.infolist():
        if info.is_dir(): continue
        destination = (WORK / info.filename).resolve()
        if WORK.resolve() not in destination.parents: raise RuntimeError(f'Unsafe payload path: {info.filename}')
        data = archive.read(info)
        if hashlib.sha256(data).hexdigest() != PAYLOAD_HASHES[info.filename]:
            raise RuntimeError(f'Embedded file checksum mismatch: {info.filename}')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)

template_path = WORK / 'training/templates/qwen3-8b.jinja'
print('Embedded source:', REPO_COMMIT)
print('Payload SHA256:', PAYLOAD_SHA256)
print('Restored files:', len(PAYLOAD_HASHES))


In [ ]:
# Load the 4-bit base at a T4-safe context and retain only complete records.
# Nothing is truncated: records above the cap are excluded as whole trajectories.
import unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from collections import Counter

BASE_MODEL = 'unsloth/Qwen3-8B'
MAX_SEQ = 4096

# Never attempt a second model load after an OOM. Colab/IPython can retain the
# failed traceback and its CUDA tensors even after del/gc/empty_cache.
free_vram, total_vram = torch.cuda.mem_get_info()
free_vram_gib = free_vram / 1024**3
if free_vram_gib < 10.0:
    raise RuntimeError(
        f'Only {free_vram_gib:.2f} GiB GPU memory is free. A previous model is still '
        'resident. Use Runtime > Disconnect and delete runtime, reopen the '
        'v7_t4_safe notebook, and Run all from a fresh runtime.'
    )
print(f'Fresh-runtime VRAM check passed: {free_vram_gib:.2f} GiB free.')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
pinned_template = template_path.read_text(encoding='utf-8')
tokenizer.chat_template = pinned_template
assert tokenizer.chat_template.strip() == pinned_template.strip()

def read_jsonl(path):
    return [
        json.loads(line)
        for line in path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]

train_rows = read_jsonl(WORK / 'datasets/frontend-stack/final/train.jsonl')
eval_rows = read_jsonl(WORK / 'datasets/frontend-stack/final/holdout.jsonl')
assert len(train_rows) == 384 and len(eval_rows) == 20
assert Counter(row['source'] for row in train_rows) == Counter({
    'frontend-stack': 230,
    'hermes-function-calling-v1': 77,
    'databricks-dolly-15k': 77,
})
assert hashlib.sha256(
    (WORK / 'datasets/frontend-stack/final/train.jsonl').read_bytes()
).hexdigest() == 'f4c36ca2b83d98137841777eb2d64024d929c82fdacfad75292607be5777d19d'
assert hashlib.sha256(
    (WORK / 'datasets/frontend-stack/final/holdout.jsonl').read_bytes()
).hexdigest() == '8d59b5c22ee6df65d8b1843d0bdf08dc352aa6640efee07d8fc6d17158485372'

def render_bounded_rows(rows, label):
    texts = []
    lengths = []
    retained_sources = []
    dropped = 0
    for row in rows:
        text = tokenizer.apply_chat_template(
            row['messages'],
            tools=row.get('tools') or None,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        if row.get('tools') and '<tools>' not in text:
            raise RuntimeError('Tool-bearing record rendered without <tools>')
        length = len(tokenizer(text, add_special_tokens=False).input_ids) + 1
        if length > MAX_SEQ:
            dropped += 1
            continue
        texts.append(text)
        lengths.append(length)
        retained_sources.append(row['source'])
    if not texts:
        raise RuntimeError(f'{label}: token cap rejected every record')
    print(
        f'{label}: retained {len(texts)}/{len(rows)} complete records; '
        f'dropped {dropped}; token range {min(lengths)}-{max(lengths)}; '
        f'sources {dict(Counter(retained_sources))}'
    )
    return Dataset.from_dict({'text': texts}), lengths

train_ds, train_lengths = render_bounded_rows(train_rows, 'train')
eval_ds, eval_lengths = render_bounded_rows(eval_rows, 'holdout')
assert max(train_lengths) <= MAX_SEQ
assert max(eval_lengths) <= MAX_SEQ
print(
    f'Guards passed: {len(train_ds)} train / {len(eval_ds)} holdout | '
    f'MAX_SEQ={MAX_SEQ}'
)


In [ ]:
from trl import SFTConfig, SFTTrainer
import gc

# PEFT 0.18.1 can detect Colab's preinstalled TorchAO, then import an API that
# some TorchAO builds do not export. This model is BitsAndBytes 4-bit, not
# TorchAO-quantized, so skip only that irrelevant PEFT dispatcher.
import peft.tuners.lora.torchao as peft_torchao_dispatch
peft_torchao_dispatch.is_torchao_available = lambda: False
print('PEFT compatibility: TorchAO dispatcher disabled (BitsAndBytes path retained).')

# If the previous failed call partially inserted an adapter, reload the clean
# 4-bit base before retrying. On a fresh Run all this branch does nothing.
partial_lora = any(hasattr(module, 'lora_A') for module in model.modules())
if partial_lora:
    print('Partial LoRA injection detected; reloading the clean base model.')
    del model
    gc.collect()
    torch.cuda.empty_cache()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ,
        load_in_4bit=True,
    )
    tokenizer.chat_template = pinned_template
    assert tokenizer.chat_template.strip() == pinned_template.strip()

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    use_gradient_checkpointing='unsloth',
    random_state=731,
)

run_config = {
    'repo_commit': REPO_COMMIT,
    'base_model': BASE_MODEL,
    'max_seq_length': MAX_SEQ,
    'train_sha256': 'f4c36ca2b83d98137841777eb2d64024d929c82fdacfad75292607be5777d19d',
    'holdout_sha256': '8d59b5c22ee6df65d8b1843d0bdf08dc352aa6640efee07d8fc6d17158485372',
    'template_sha256': '0958f6ebb716badab20903885efa1a26ee5a8d2ce9162bfcfe982ff43cff4077',
    'epochs': 2.0,
    'learning_rate': 2e-4,
    'effective_batch': 16,
    'save_steps': 1,
    'seed': 731,
}

config_path = DRIVE_ROOT / f'run_config-{MAX_SEQ}.json'
if config_path.exists():
    if json.loads(config_path.read_text()) != run_config:
        raise RuntimeError(
            'Checkpoint config differs from this notebook; use a new DRIVE_ROOT.'
        )
else:
    config_path.write_text(json.dumps(run_config, indent=2), encoding='utf-8')

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        output_dir=str(TRAINER_OUT),
        dataset_text_field='text',
        max_length=MAX_SEQ,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        num_train_epochs=2.0,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        logging_steps=1,
        eval_strategy='steps',
        eval_steps=8,
        save_strategy='steps',
        save_steps=1,
        save_total_limit=2,
        optim='paged_adamw_8bit',
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        weight_decay=0.01,
        seed=731,
        report_to='none',
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

def checkpoint_step(path):
    try:
        return int(path.name.rsplit('-', 1)[-1])
    except ValueError:
        return -1

def checkpoint_is_complete(path):
    required = [
        path / 'trainer_state.json',
        path / 'optimizer.pt',
        path / 'scheduler.pt',
    ]
    weights = [
        path / 'adapter_model.safetensors',
        path / 'adapter_model.bin',
        path / 'model.safetensors',
        path / 'pytorch_model.bin',
    ]
    if not all(item.is_file() and item.stat().st_size > 0 for item in required):
        return False
    if not any(item.is_file() and item.stat().st_size > 0 for item in weights):
        return False
    try:
        json.loads((path / 'trainer_state.json').read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return False
    return True

checkpoint_candidates = sorted(
    TRAINER_OUT.glob('checkpoint-*'),
    key=checkpoint_step,
    reverse=True,
)
resume_checkpoint = next(
    (str(path) for path in checkpoint_candidates if checkpoint_is_complete(path)),
    None,
)
skipped_checkpoints = [
    str(path) for path in checkpoint_candidates if not checkpoint_is_complete(path)
]
if skipped_checkpoints:
    print('Ignoring incomplete checkpoints:', skipped_checkpoints)
print('Resume checkpoint:', resume_checkpoint or 'none (fresh run)')


In [ ]:
# Long-running cell. Re-running the notebook resumes from the newest Drive checkpoint.
stats = trainer.train(resume_from_checkpoint=resume_checkpoint)
model.save_pretrained(str(LORA_OUT), safe_serialization=True)
tokenizer.save_pretrained(str(LORA_OUT))
completion = {
    **run_config, 'completed_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'gpu': gpu.name, 'vram_gib': round(vram_gib, 3), 'metrics': stats.metrics,
    'packages': PACKAGE_VERSIONS, 'torch': torch.__version__, 'notebook_revision': NOTEBOOK_REVISION,
}
(DRIVE_ROOT / 'TRAINING_COMPLETE.json').write_text(json.dumps(completion, indent=2, default=str), encoding='utf-8')
print('Training complete. LoRA:', LORA_OUT)
print(json.dumps(stats.metrics, indent=2, default=str))
# Release paged optimizer/scheduler state before inference and GGUF merge.
import gc
trainer.optimizer = None
trainer.lr_scheduler = None
del trainer
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cheap pre-export checks: one pi-schema tool call and one forgetting check.
FastLanguageModel.for_inference(model)
pi_tools = json.loads((WORK / 'training/pi_tools.json').read_text(encoding='utf-8'))
tool_messages = [
    {'role':'system','content':'You are an expert coding assistant operating inside pi. Read before editing and use only provided tools.'},
    {'role':'user','content':'Fix the disabled SaveButton in src/SaveButton.tsx.'},
]
prompt = tokenizer.apply_chat_template(tool_messages, tools=pi_tools, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to('cuda')
generated = model.generate(input_ids=prompt, max_new_tokens=256, do_sample=False, use_cache=True)
tool_text = tokenizer.decode(generated[0, prompt.shape[1]:], skip_special_tokens=False)
print('TOOL SMOKE:')
print(tool_text)
tool_smoke_pass = '<tool_call>' in tool_text or '<function=' in tool_text

plain = tokenizer.apply_chat_template([{'role':'user','content':'What is 17 multiplied by 19? Answer with the number only.'}], tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to('cuda')
plain_out = model.generate(input_ids=plain, max_new_tokens=32, do_sample=False, use_cache=True)
plain_text = tokenizer.decode(plain_out[0, plain.shape[1]:], skip_special_tokens=True).strip()
print('MATH SMOKE:', plain_text)
math_smoke_pass = '323' in plain_text
smoke = {'tool_smoke_pass': tool_smoke_pass, 'math_smoke_pass': math_smoke_pass, 'tool_output': tool_text, 'math_output': plain_text}
(DRIVE_ROOT / 'SMOKE_TEST.json').write_text(json.dumps(smoke, indent=2), encoding='utf-8')
if not all((tool_smoke_pass, math_smoke_pass)): print('WARNING: smoke test failed; GGUF will still export for the full local gate.')

In [ ]:
# Export or recover the Q4_K_M GGUF, copy it to the final output, verify it,
# clean temporary merge files, and start the download when Drive is disabled.
import gc
import hashlib
import json
import os
import shutil
from pathlib import Path

# Unsloth uses SAVE_DIRECTORY for merged HF weights, but writes GGUF files to
# SAVE_DIRECTORY + "_gguf". The previous cell searched the wrong directory.
MERGED_DIR = Path('/content/frontend-stack-gguf')
UNSLOTH_GGUF_DIR = Path(str(MERGED_DIR) + '_gguf')

def discover_ggufs(*roots):
    discovered = {}
    for root in roots:
        root = Path(root)
        if root.is_file() and root.suffix.lower() == '.gguf':
            discovered[str(root.resolve())] = root.resolve()
        elif root.is_dir():
            for candidate in root.rglob('*'):
                if candidate.is_file() and candidate.suffix.lower() == '.gguf':
                    discovered[str(candidate.resolve())] = candidate.resolve()
    return sorted(discovered.values(), key=lambda path: str(path).lower())

def is_q4_k_m(path):
    normalized = path.name.lower().replace('-', '_').replace('.', '_')
    return 'q4_k_m' in normalized

def valid_gguf(path):
    if not path.is_file() or path.stat().st_size < 1024**3:
        return False
    with path.open('rb') as handle:
        return handle.read(4) == b'GGUF'

# A previous export can be complete even though the old discovery check failed.
all_ggufs = discover_ggufs(UNSLOTH_GGUF_DIR, MERGED_DIR)
q4_ggufs = [path for path in all_ggufs if is_q4_k_m(path) and valid_gguf(path)]
reused_previous_export = len(q4_ggufs) == 1

if reused_previous_export:
    src_gguf = q4_ggufs[0]
    print('Reusing completed GGUF from the previous cell:', src_gguf)
else:
    if q4_ggufs:
        raise RuntimeError(f'Expected one valid Q4_K_M GGUF, found: {q4_ggufs}')

    # No usable prior Q4 file. Remove only these known temporary export folders,
    # then perform one clean conversion and trust Unsloth's returned file paths.
    for temporary in (MERGED_DIR, UNSLOTH_GGUF_DIR):
        if temporary.exists():
            shutil.rmtree(temporary)
    free_gib = shutil.disk_usage('/content').free / 1024**3
    if free_gib < 22:
        raise RuntimeError(
            f'Only {free_gib:.1f} GiB disk is free; clean the Colab runtime or '
            'enable Drive before repeating the 8B merge/quantization.'
        )

    MERGED_DIR.mkdir(parents=True)
    gc.collect()
    torch.cuda.empty_cache()
    export_result = model.save_pretrained_gguf(
        str(MERGED_DIR),
        tokenizer,
        quantization_method='q4_k_m',
        maximum_memory_usage=0.5,
    )
    print('Unsloth export directory:', export_result.get('gguf_directory'))
    returned_paths = [Path(path) for path in export_result.get('gguf_files', [])]
    all_ggufs = discover_ggufs(
        *returned_paths,
        Path(export_result.get('gguf_directory', UNSLOTH_GGUF_DIR)),
        UNSLOTH_GGUF_DIR,
        MERGED_DIR,
    )
    q4_ggufs = [path for path in all_ggufs if is_q4_k_m(path) and valid_gguf(path)]
    if len(q4_ggufs) != 1:
        raise RuntimeError(
            'GGUF conversion returned, but exactly one valid Q4_K_M artifact was '
            f'not found. Returned={returned_paths}; discovered={all_ggufs}'
        )
    src_gguf = q4_ggufs[0]

GGUF_OUT.mkdir(parents=True, exist_ok=True)
dst_gguf = GGUF_OUT / 'frontend-stack-qwen3-8b-q4_k_m.gguf'
partial_gguf = dst_gguf.with_suffix(dst_gguf.suffix + '.partial')
if partial_gguf.exists():
    partial_gguf.unlink()
shutil.copy2(src_gguf, partial_gguf)
if partial_gguf.stat().st_size != src_gguf.stat().st_size:
    raise RuntimeError('GGUF copy size mismatch')
with partial_gguf.open('rb') as handle:
    if handle.read(4) != b'GGUF':
        raise RuntimeError('Copied artifact does not have a GGUF header')
os.replace(partial_gguf, dst_gguf)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

artifact = {
    'path': str(dst_gguf),
    'bytes': dst_gguf.stat().st_size,
    'sha256': sha256_file(dst_gguf),
    'quantization': 'Q4_K_M',
    'reused_previous_export': reused_previous_export,
    'storage': 'google-drive' if DRIVE_AVAILABLE else 'temporary-runtime',
    'notebook_revision': NOTEBOOK_REVISION,
}
(GGUF_OUT / 'artifact.json').write_text(
    json.dumps(artifact, indent=2),
    encoding='utf-8',
)
print(json.dumps(artifact, indent=2))

# The final verified copy is independent, so remove large temporary merged and
# quantization folders to reclaim Colab disk space.
for temporary in (MERGED_DIR, UNSLOTH_GGUF_DIR):
    if temporary.exists() and temporary.resolve() not in dst_gguf.resolve().parents:
        shutil.rmtree(temporary)
print('Temporary export directories cleaned.')

if DRIVE_AVAILABLE:
    print('Done. Verified GGUF is syncing to Google Drive.')
else:
    print('Starting browser download. Keep this tab open until it finishes.')
    from google.colab import files
    files.download(str(dst_gguf))


## Outputs

Default: temporary `/content/Master-Models-Colab/frontend-stack-qwen3-8b/`; the final cell automatically starts the GGUF download. Keep the Colab tab open until it finishes.

With `USE_GOOGLE_DRIVE = True`: `MyDrive/Master-Models-Colab/frontend-stack-qwen3-8b/`.

- `trainer/`: two latest resumable checkpoints
- `lora/`: final adapter and exact tokenizer/template
- `gguf/frontend-stack-qwen3-8b-q4_k_m.gguf`: final model
- `TRAINING_COMPLETE.json`, `SMOKE_TEST.json`, and `gguf/artifact.json`: training, smoke-test, and SHA-256 metadata
